# Bayesian optimization of group sequential designs

## Motivating problem

### Introduction

In Bayesian optimization, the goal is to optimize a blackbox function, $f$. In our case, this function takes inputs of 
$$
D = \{n, u_1, \ell_1, u_2, \ell_2, \cdots, u_k, \ell_k \}
$$

where 

- $D$ is the design (an $n$-tuple set),
- $n$ is the sample size at each analysis point (total for both groups),
- $u_1$ and $\ell_1$ are the upper and lower bounds for the first stage, and
- $k$ is the total number of stages

and outputs several values of interest, $T$ (an $m$-tuple set), including (but not limited to)

- $\alpha$, the type I error
- $\beta$, the type II error
- $\mathbb{E}[N \, | \, \boldsymbol{\delta}]$, the expected sample size (ESS) $N$ over a range of differences, $\delta$, between groups $\boldsymbol{\delta} = \{\delta_1, \delta_2, \dots, \delta_j \}$

Above, $N = k*n$ and 

The expected sample size is calculated across a set of possible true treatment effects $\boldsymbol{\delta} = \{\delta_1, \delta_2, \dots, \delta_j \}$ as a function of the design elements $D$: (1) number of analyses $\{1, 2, \dots, k\}$, (2) number of patients at each analysis $\mathbf{n} = \{n_1, n_2, \dots, n_k\}$, and (3) the upper and lower bounds $\mathbf{u} = (u_1, u_2, \dots, u_k)$ and $\boldsymbol{\ell} = (\ell_1, \ell_2, \dots, \ell_k)$.

It is calculated as
$$
\mathbb{E}[N \, | \, \boldsymbol{\delta}]=\sum_{i=1}^k n_i P(\text{trial stops after analysis }i \, | \, \boldsymbol{\delta})
$$

### Pseudocode for Bayesian optimization loop

In order to generated an optimized clinical trial design, the following steps must be followed:

1. Generate several feasible trial designs, $\boldsymbol{D}=\{D_1, D_2, \cdots, D_p\}$.
2. Generate the corresponding outputs, $\boldsymbol{T}=\{T_1, T_2, \cdots, T_p\}$.
3. Fit a Gaussian process regression model to estimate the blackbox function $f: D \to T$.
4. Perform a step of Bayesian optimization to find the next trial design of interest $D_i$.
5. Obtain the corresponding outputs that correspond to this design $T_i$.
6. Refit the Gaussian process regression model on the new data $n$-tuples $\{(\boldsymbol{D}, \boldsymbol{T}), (D_i, T_i)\}$

Repeat until termination policy is reached.

### Function to minimize

As the above optimization problem applies to clinical trial designs, there are certain feasibility constraints that must be considered. Most importantly, the type I and type II error (or power)&mdash;$\alpha$, $\beta$ (or $1-\beta$),respectively&mdash;must be near the set nominal levels. In other words, if the design requires that a one-sided $\alpha = 0.025$, then feasible designs must have values of $\alpha$ near this value. In Wason et al. (Statist. Med. 2012, 31 301–312), feasible designs are defined as "design[s] for which the significance level and power meet the required constraints."

Though Bayesian optimization can be constrained in such a manner, for simplicity, a penalty term will be included within the objective function $f$, which will take these constraints into account. Again, borrowing from Wason et al., the penalty term is:

$$
\mathcal{L} = \mu \cdot \left( \mathbb{I}_{\{\alpha' > \alpha\}}\cdot\frac{\alpha' - \alpha}{\alpha} + \mathbb{I}_{\{\beta' > \beta\}}\cdot\frac{\beta' - \beta}{\beta}  \right)
$$

where $\mu$ is the sample size for a one-stage design, $\alpha$ and $\beta$ are the set nominal values for type I and II error, respectively, $\alpha'$ and $\beta'$ are the type I and II errors for the new design, and $\mathbb{I}$ is the indicator function.

The function that we aim to minimise, $f$, is the sum of the maximum expected sample size of the design and a penalty function that penalizes designs that are not considered feasible:

$$
f = \max\left\{\mathbb{E}[N \, | \, \boldsymbol{\delta}]\right\} + \mathcal{L}
$$

## Implementation of Bayesian optimization loop

In [182]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [11]:
# higher resolution graphs
%config InlineBackend.figure_format='retina'

### Step 1: Generate study designs

In [281]:
# simulate the trials to obtain alpha and beta
def simulate_group_sequential_designs(
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        n_patients=[20],
        null_hypothesis=0,
        alt_hypothesis=0.5,
        variance=1):

    # assign values for null and alt hypotheses
    theta_0 = null_hypothesis
    delta = alt_hypothesis

    # empty list to fill mean vectors
    mean_0 = []
    mean_1 = []

    # number of patients in each analysis
    n_patients_analysis = np.array([x for x in range(1, n_analyses + 1, 1)]) * n_patients

    # need to parse the upper and lower boundaries of the design
    # for futility and efficacy, must put the bounds of integration correctly
    # for pmvnorm
    futility_l_bounds = [[]]
    futility_u_bounds = [[]]
    efficacy_l_bounds = [[]]
    efficacy_u_bounds = [[]]

    n_analyses = len(upper_bounds)

    # loop through number of analyses
    for i in range(n_analyses):

        # special case of i = 1
        if i == 0:
            futility_l_bounds[i].append(-np.inf)
            futility_u_bounds[i].append(lower_bounds[i])
            efficacy_l_bounds[i].append(upper_bounds[i])
            efficacy_u_bounds[i].append(np.inf)
            continue

        # all other cases
        futility_l_bounds.append(lower_bounds[0:i] + [-np.inf])
        futility_u_bounds.append(upper_bounds[0:i] + [lower_bounds[i]])
        efficacy_l_bounds.append(lower_bounds[0:i] + [upper_bounds[i]])
        efficacy_u_bounds.append(upper_bounds[0:i] + [np.inf])

    # empty dictionary of SIGMA matrices
    SIGMA_dict = dict()

    # generate the SIGMA matrices
    for i in range(n_analyses):

        if i == 0: SIGMA_dict.update({i : np.sqrt(variance)})

        # start with diagonal matrix for SIGMA
        SIGMA = np.eye(N = i+1)

        # n = 2, need to fill all but 11, 22
        # n = 3, need to fill all but 11, 22, 33
        # n = 4, need to fill all but 11, 22, 33, 44
        # etc.
        for j in range(i+1):
            for k in range(i+1):

                # leave the 1s on the diagonal, skip interation
                if j == k: continue

                # when j is less than k, the lower number of patients will be in numerator
                if j < k: SIGMA[j,k] = np.sqrt(n_patients_analysis[j] / n_patients_analysis[k])

                # when j is greater than j, the lower number of patients will be in numerator
                if j > k: SIGMA[j,k] = np.sqrt(n_patients_analysis[k] / n_patients_analysis[j])

        SIGMA_dict.update({i : SIGMA})

    # empty data frame to collect probabilities
    # dictionary to DataFrame, column is the key:value pair of the dictionary
    probs_to_return = {
        "futility_null" : np.empty(n_analyses),
        "efficacy_null" : np.empty(n_analyses),
        "futility_alt" : np.empty(n_analyses),
        "efficacy_alt" : np.empty(n_analyses)
    }

    # generate the row names
    row_names = [ana + str(row_names) for row_names in range(1, n_analyses + 1, 1)]

    # create the empty data frame with data and index
    probs_to_return = pd.DataFrame(data = probs_to_return, index = row_names)
    
    # start calculations for the analyses
    for i in range(n_analyses):
        
        # mean under null
        mean_0.append(theta_0 * np.sqrt(n_patients_analysis[i] / (2 * variance)))

        # mean under alternative
        mean_1.append(delta   * np.sqrt(n_patients_analysis[i] / (2 * variance)))
        
        # generate the null and alt multivariate normal
        mvn_null = stats.multivariate_normal(mean = mean_0, cov = SIGMA_dict[i])
        mvn_alt  = stats.multivariate_normal(mean = mean_1, cov = SIGMA_dict[i])
        
        # prob stop for futility under null
        futility_null = mvn_null.cdf(futility_u_bounds[i], lower_limit = futility_l_bounds[i])
        
        # prob stop for futility under alt
        futility_alt = mvn_alt.cdf(futility_u_bounds[i], lower_limit = futility_l_bounds[i])
        
        # prob stop for efficacy under null
        efficacy_null = mvn_null.cdf(efficacy_u_bounds[i], lower_limit = efficacy_l_bounds[i])

        # prob stop for efficacy under alt
        efficacy_alt = mvn_alt.cdf(efficacy_u_bounds[i], lower_limit = efficacy_l_bounds[i])

        # add them to the data frame to return
        probs_to_return.iloc[i] = [futility_null, efficacy_null, futility_alt, efficacy_alt]

    # get the type I error (alpha)
    alpha = probs_to_return.sum(axis = 0)["efficacy_null"]

    # get the power (1 - beta)
    power = probs_to_return.sum(axis = 0)["efficacy_alt"]

    # get expected sample size
    
    return probs_to_return, alpha, power
        

Are the diagonals of the covariance matrix supposed to be the variance? Or are they always 1?

In [282]:
simulate_group_sequential_designs()

(            futility_null  efficacy_null  futility_alt  efficacy_alt
 analysis_1       0.500000       0.006210      0.056923      0.179084
 analysis_2       0.298776       0.019415      0.042323      0.419684
 analysis_3       0.137285       0.038314      0.049028      0.252958,
 0.06393887320195822,
 0.851725380616087)

Test speed of multivariate integration vs. single variable integration

In [146]:
mvn = stats.multivariate_normal(mean = [1, 2], cov = [[1, 0.5], [0.5, 1]])

In [149]:
mvn.cdf([2, np.inf], lower_limit=[-np.inf, -np.inf])

0.8413447460685429

In [161]:
stats.norm.cdf(2, 1, 1)

0.8413447460685429

In [162]:
%%timeit
stats.norm.cdf(2, 1, 1)

16.7 μs ± 296 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [163]:
%%timeit
mvn.cdf([2, np.inf], lower_limit=[-np.inf, -np.inf])

17 μs ± 241 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Functions for 3 different study designs.

In [ ]:
# Pocock boundaries
def calculate_pocock_boundaries:

In [ ]:
# O'Brien-Fleming boundaries
def calculate_of_boundaries:

In [ ]:
# Triangular boundaries
def calculate_triangular_boundaries:

### Step 2: Simulate the trials to obtain $\alpha$, $\beta$, maximum ESS

In [ ]:
# obtain maximum expected sample size
def max_ess:

In [ ]:
# generate the penalty term
def feasibility_penalty:

### Step 3: Fit a Gaussian process regression model

In [ ]:
# function to minimize
def function_to_minimize:

In [ ]:
# imports

### Steps 4-6: Bayesian optimization loop

In [ ]:
# imports